# Glyph — Генерация synthetic instruction dataset

Для каждого автора LLM генерирует ~500 пар (вопрос, ответ в стиле автора) на основе случайно выбранных фрагментов корпуса. Результат используется для instruction-tuning LoRA (этап C).

## Подготовка API ключа

**Groq** (рекомендуется — бесплатно):
1. Зарегистрируйся на https://console.groq.com/keys и создай API Key
2. Слева в Colab — иконка 🔑 **Secrets** → Add new secret:
   - Name: `GROQ_API_KEY`
   - Value: твой ключ
   - Enable Notebook access: ✅

**Gemini** (альтернатива, также бесплатно):
- Secret name: `GEMINI_API_KEY` (получить на https://aistudio.google.com/apikey)

**Anthropic** (платно, ~$9 за датасет):
- Secret name: `ANTHROPIC_API_KEY` (получить на https://console.anthropic.com)

## Что делает
1. Клонирует репозиторий Glyph
2. Устанавливает httpx
3. Читает выбранный API-ключ из Colab Secrets
4. Запускает `generate_synthetic_dataset.py` по очереди для 3 авторов
5. Архивирует `ml/data/synthetic/*.jsonl` и даёт скачать

## Оценка времени (с учётом rate limits)
- **Groq free** (llama-3.1-8b-instant, RPM=12): ~40 мин на автора × 3 = **~2 часа**
- **Gemini free** (RPM=15): ~35 мин на автора × 3 = ~1.5 часа
- **Anthropic Haiku** (платно, RPM=50): ~10 мин на автора × 3 = ~30 мин

Скрипт идемпотентен — переживает обрывы Colab, просто запусти заново.

## 1. Клонирование и установка

In [ ]:
%cd /content
!rm -rf glyph
!git clone --depth 1 https://github.com/tearfu1/glyph.git
%cd /content/glyph/ml
!pip install -q httpx

## 2. Выбор провайдера и ключа

Отредактируй `PROVIDER` ниже, если нужно.

In [ ]:
import os
from google.colab import userdata

# Какой провайдер используем
PROVIDER = "groq"  # варианты: "groq", "gemini", "anthropic"

SECRET_NAME = {
    "groq": "GROQ_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}[PROVIDER]

ENV_NAME = SECRET_NAME
os.environ[ENV_NAME] = userdata.get(SECRET_NAME)
os.environ["LLM_PROVIDER"] = PROVIDER

print(f"Провайдер: {PROVIDER}")
print(f"Ключ установлен: {ENV_NAME} (длина {len(os.environ[ENV_NAME])} символов)")

## 3. Быстрый тест

Сгенерируем 3 пары на Достоевском — проверим что ключ работает и формат JSON корректно парсится.

In [ ]:
!python -m scripts.generate_synthetic_dataset \
    --author dostoevsky \
    --num-samples 3 \
    --provider {PROVIDER}

!head -3 data/synthetic/dostoevsky.jsonl

## 4. Полная генерация для всех 3 авторов

По 500 пар на автора. Скрипт идемпотентный — если упадёт, запускай заново, продолжит с места остановки (читает уже записанное из JSONL).

In [ ]:
NUM_SAMPLES = 500  # на автора
RPM = 12  # Groq free tier llama-3.1-8b-instant (TPM 30k)

for author in ["dostoevsky", "chekhov", "bulgakov"]:
    print(f"\n{'='*60}\n=== Автор: {author} ===\n{'='*60}")
    !python -m scripts.generate_synthetic_dataset \
        --author {author} \
        --num-samples {NUM_SAMPLES} \
        --provider {PROVIDER} \
        --rpm {RPM}

## 5. Проверка результата

In [ ]:
import json

for author in ["dostoevsky", "chekhov", "bulgakov"]:
    path = f"data/synthetic/{author}.jsonl"
    with open(path) as f:
        records = [json.loads(l) for l in f]
    print(f"\n{author}: {len(records)} пар")
    # Показываем первую пару как пример
    if records:
        r = records[0]
        print(f"  Q: {r['question']}")
        print(f"  A: {r['answer'][:200]}...")

## 6. Скачать архив

Распакуй локально в `ml/data/synthetic/`.

In [ ]:
!cd data && zip -r /content/glyph_synthetic_dataset.zip synthetic/
from google.colab import files
files.download("/content/glyph_synthetic_dataset.zip")